# Product Category Translation - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.olist_product_category_translation"
target_table = f"{catalog}.silver.olist_product_category_translation"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(bronze_df.limit(10))

product_category_name,product_category_name_english,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
beleza_saude,health_beauty,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
informatica_acessorios,computers_accessories,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
automotivo,auto,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
cama_mesa_banho,bed_bath_table,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
moveis_decoracao,furniture_decor,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
esporte_lazer,sports_leisure,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
perfumaria,perfumery,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
utilidades_domesticas,housewares,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
telefonia,telephony,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
relogios_presentes,watches_gifts,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation


In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

Number of rows: 71
Number of columns: 9


In [0]:
null_count_product_category_name = bronze_df.filter(col("product_category_name").isNull()).count()

null_count_product_category_name_english = bronze_df.filter(col("product_category_name_english").isNull()).count()

print("Null count for product_category_name:", null_count_product_category_name)
print("Null count for product_category_name_english:", null_count_product_category_name_english)

Null count for product_category_name: 0
Null count for product_category_name_english: 0


Both columns have no null values.

In [0]:
distinct_count_product_category_name = bronze_df.select("product_category_name").distinct().count()

distinct_count_product_category_name_english = bronze_df.select("product_category_name_english").distinct().count()

print("Distinct count for product_category_name:", distinct_count_product_category_name)
print("Distinct count for product_category_name_english:", distinct_count_product_category_name_english)

Distinct count for product_category_name: 71
Distinct count for product_category_name_english: 71


- There are no duplicate values in both columns.
- Since this is a translation mapping table, we will treat product_category_name as the table key.

In [0]:
rescued_row_count = (
    bronze_df
    .filter(col("_rescued_data").isNotNull())
    .filter(trim(col("_rescued_data")) != "")
    .count()
)

print("Number of rescued rows:", rescued_row_count)

Number of rescued rows: 0


There are no rescued data.

In [0]:
display(
    bronze_df.withColumn("product_category_name_trimmed", trim(col("product_category_name")))
    .filter(col("product_category_name") != col("product_category_name_trimmed"))
    .count()
)

0

In [0]:
display(
    bronze_df.withColumn("product_category_name_english_trimmed", trim(col("product_category_name_english")))
    .filter(col("product_category_name_english") != col("product_category_name_english_trimmed"))
    .count()
)

0

Both columns do not contain extra whitespace.

## Transform to Silver

In [0]:
silver_df = bronze_df.withColumnRenamed(
    "product_category_name",
    "product_category_name_portuguese"
)

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

root
 |-- product_category_name_portuguese: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(silver_table_df.limit(10))

product_category_name_portuguese,product_category_name_english,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
beleza_saude,health_beauty,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
informatica_acessorios,computers_accessories,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
automotivo,auto,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
cama_mesa_banho,bed_bath_table,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
moveis_decoracao,furniture_decor,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
esporte_lazer,sports_leisure,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
perfumaria,perfumery,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
utilidades_domesticas,housewares,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
telefonia,telephony,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation
relogios_presentes,watches_gifts,null,/Volumes/ecommerce_dev/landing/raw_files/olist/product_category_translation/product_category_name_translation.csv,2026-08-02T21:31:00.000Z,2026-08-02T22:59:55.294Z,1be1bd03-3a3f-4d33-8ff1-27f3093a5220,olist,product_category_translation


In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())

Bronze row count: 71
Silver row count: 71
